# Step 10: Final Testing & Submission Preparation

This notebook:
1. Runs all Must-Have questions end-to-end with tracing
2. Evaluates routing accuracy and answer quality
3. Generates a comprehensive results summary
4. Prepares project documentation for submission

**Run notebooks 00-09 first!**

## 10A. Install Packages

In [0]:
%pip install -U typing-extensions pydantic
%pip install langchain langchain-community databricks-vectorsearch mlflow sentence-transformers faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 988.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.4.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-2d2ff2a5-4b59-414f-ab9c-ceff063b3e04
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.6
    Not uninstalling pydantic at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-2d2ff2a5-4b59-414f-ab9c-ceff063b3e04
    Can't uninstall 'pydantic'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python 

In [0]:
dbutils.library.restartPython()

## 10B. Configuration

In [0]:
import os, json, time, mlflow
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from langchain_community.chat_models import ChatDatabricks
from langchain_core.prompts import ChatPromptTemplate
from databricks.vector_search.client import VectorSearchClient

CATALOG = "hackathon_vf"
SCHEMA = "healthcare"
TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"
try:
    spark.sql(f"USE CATALOG {CATALOG}")
    spark.sql(f"USE SCHEMA {SCHEMA}")
except Exception:
    CATALOG = "hive_metastore"
    SCHEMA = "hackathon"
    TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"
    spark.sql(f"USE {SCHEMA}")

ENRICHED_TABLE = f"{TABLE_PREFIX}.facilities_enriched"
DESERT_TABLE = f"{TABLE_PREFIX}.regional_analysis"
VS_ENDPOINT = "vf_facility_search"
VS_INDEX_NAME = f"{TABLE_PREFIX}.facilities_vs_index"
RESULTS_TABLE = f"{TABLE_PREFIX}.evaluation_results"

try:
    DATABRICKS_TOKEN = dbutils.secrets.get(scope="hackathon", key="DATABRICKS_TOKEN")
except Exception:
    DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN", "YOUR_DATABRICKS_TOKEN_HERE")
os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN

llm = ChatDatabricks(model="llama-3.3-70b-versatile", temperature=0, max_tokens=2048)
experiment_name = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/vf_healthcare_agent"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2016804619128450', creation_time=1774796984983, experiment_id='2016804619128450', last_update_time=1775121045878, lifecycle_stage='active', name='/Users/vm8810@srmist.edu.in/vf_healthcare_agent', tags={'mlflow.experiment.sourceName': '/Users/vm8810@srmist.edu.in/vf_healthcare_agent',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'vm8810@srmist.edu.in',
 'mlflow.ownerId': '70724061467260'}, workspace='default'>

## 10C. Load All Agent Components

In [0]:
# Vector Search
vsc = VectorSearchClient()
USING_VS = False
vs_index = None
try:
    vs_index = vsc.get_index(VS_ENDPOINT, VS_INDEX_NAME)
    USING_VS = True
    print("Vector Search connected")
except Exception:
    try:
        import faiss, pickle, tempfile, numpy as np
        from sentence_transformers import SentenceTransformer
        local_dir = tempfile.mkdtemp()
        dbutils.fs.cp("/FileStore/hackathon/vector_store/facility_index.faiss", f"file:{local_dir}/facility_index.faiss")
        dbutils.fs.cp("/FileStore/hackathon/vector_store/facility_metadata.pkl", f"file:{local_dir}/facility_metadata.pkl")
        faiss_index = faiss.read_index(f"{local_dir}/facility_index.faiss")
        with open(f"{local_dir}/facility_metadata.pkl", "rb") as f:
            faiss_meta = pickle.load(f)
        embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        print(f"FAISS loaded: {faiss_index.ntotal} vectors")
    except Exception as e:
        print(f"No vector search available: {e}")

facilities_pdf = spark.table(ENRICHED_TABLE).toPandas()
print(f"Loaded {len(facilities_pdf)} facilities")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Vector Search connected
Loaded 797 facilities


## 10D. Sub-Agent Functions

In [0]:
# Compact sub-agents (same as notebook 09)
CLASSIFY_PROMPT = ChatPromptTemplate.from_template(
    "Classify into ONE: STRUCTURED, SEMANTIC, REASONING, DESERT. Question: {question}\nCategory:")

def classify_query(q):
    cat = llm.invoke(CLASSIFY_PROMPT.format(question=q)).content.strip().upper()
    valid = {"STRUCTURED","SEMANTIC","REASONING","DESERT"}
    return cat if cat in valid else next((v for v in valid if v in cat), "SEMANTIC")

def sql_sub_agent(q):
    p = ChatPromptTemplate.from_template(f"Spark SQL SELECT only, no backticks. Table: {ENRICHED_TABLE}. "
        f"Cols: pk_unique_id, name, facilityTypeId, address_stateOrRegion, numberDoctors, capacity, "
        f"specialties, procedure, equipment, num_procedures, num_equipment, num_specialties, "
        f"flag_procedures_no_doctors, flag_capacity_no_equipment. Regional: {DESERT_TABLE}. "
        "LOWER(col) LIKE '%term%'. COALESCE nulls. Q: {{q}}\nSQL:")
    sql = llm.invoke(p.format(q=q)).content.strip()
    if sql.startswith("```"): sql = "\n".join(l for l in sql.split("\n") if not l.strip().startswith("```"))
    sql = sql.strip().rstrip(";")
    for kw in ["DROP","DELETE","UPDATE","INSERT","ALTER","CREATE"]:
        if kw in sql.upper(): return f"BLOCKED", sql, []
    try:
        rows = spark.sql(sql).limit(30).collect()
        cols = spark.sql(sql).columns
        return "\n".join(" | ".join(str(r[c]) for c in cols) for r in rows), sql, []
    except Exception as e:
        return f"Error: {e}", sql, []

def rag_sub_agent(q):
    if not (USING_VS and vs_index): return "Vector search unavailable.", [], []
    res = vs_index.similarity_search(query_text=q, columns=["pk_unique_id","name","search_text","address_stateOrRegion","facilityTypeId","specialties","numberDoctors"], num_results=10)
    cols = [c["name"] for c in res["manifest"]["columns"]]
    facs = [dict(zip(cols, row)) for row in res["result"]["data_array"]]
    sids = [f.get("pk_unique_id","") for f in facs]
    ctx = "\n".join(f"[{i+1}] {f.get('name','?')} ({f.get('facilityTypeId','?')}) {f.get('address_stateOrRegion','?')} Specs:{f.get('specialties','[]')}" for i,f in enumerate(facs))
    ans = llm.invoke(ChatPromptTemplate.from_template("Healthcare agent. Answer from data, cite names.\nDATA:\n{c}\nQ: {q}\nA:").format(c=ctx,q=q)).content
    return ans, sids, facs

def reasoning_sub_agent(q):
    rel = facilities_pdf.sort_values("source_count", ascending=False).head(15)
    txt = "\n".join(f"- {r.get('name','?')} [{r.get('facilityTypeId','?')}] {r.get('address_stateOrRegion','?')} Docs:{r.get('numberDoctors','N/A')} Procs:{r.get('num_procedures',0)} Equip:{r.get('num_equipment',0)}" for _,r in rel.iterrows())
    return llm.invoke(ChatPromptTemplate.from_template("Medical reasoning.\nDATA:\n{d}\nQ: {q}\nA:").format(d=txt,q=q)).content, []

def desert_sub_agent(q):
    try:
        rows = spark.table(DESERT_TABLE).orderBy("desert_score", ascending=False).collect()
        data = "\n".join(f"{r['address_stateOrRegion']}: Fac={r['facility_count']}, Docs={r['total_doctors']}, Surgery={'Y' if r['has_surgery']>0 else 'N'}, Score={r['desert_score']}" for r in rows)
    except: data = "N/A"
    return llm.invoke(ChatPromptTemplate.from_template("Desert analyst.\nDATA:\n{d}\nQ: {q}\nA:").format(d=data,q=q)).content, []

def traced_query(question):
    t0 = time.time()
    category = classify_query(question)
    if category=="STRUCTURED": raw, sql, sids = sql_sub_agent(question); agent="sql"
    elif category=="SEMANTIC": raw, sids, _ = rag_sub_agent(question); agent="rag"
    elif category=="REASONING": raw, sids = reasoning_sub_agent(question); agent="reasoning"
    else: raw, sids = desert_sub_agent(question); agent="desert"
    if category=="STRUCTURED" and not raw.startswith(("BLOCKED","Error")):
        final = llm.invoke(ChatPromptTemplate.from_template("Answer naturally.\nQ: {q}\nData:\n{d}\nA:").format(q=question,d=raw[:3000])).content
    else: final = raw
    return {"question": question, "category": category, "agent": agent, "answer": final, "time": round(time.time()-t0,2)}

## 10E. Full Must-Have Test Suite

In [0]:
MUST_HAVE_QUESTIONS = [
    {"id": "1.1", "question": "How many hospitals have cardiology?", "expected": "STRUCTURED", "category": "Basic Queries"},
    {"id": "1.2", "question": "How many hospitals in Greater Accra have the ability to perform surgery?", "expected": "STRUCTURED", "category": "Basic Queries"},
    {"id": "1.3", "question": "What services does Korle Bu Teaching Hospital offer?", "expected": "SEMANTIC", "category": "Basic Queries"},
    {"id": "1.4", "question": "Are there any clinics in Tamale that do surgery?", "expected": "SEMANTIC", "category": "Basic Queries"},
    {"id": "1.5", "question": "Which region has the most hospital-type facilities?", "expected": "STRUCTURED", "category": "Basic Queries"},
    {"id": "2.1", "question": "Which regions have hospitals that can treat cardiac conditions?", "expected": "STRUCTURED", "category": "Geospatial"},
    {"id": "2.3", "question": "Where are the largest geographic cold spots where critical procedures are absent?", "expected": "DESERT", "category": "Geospatial"},
    {"id": "4.4", "question": "Which facilities claim an unrealistic number of procedures relative to their size?", "expected": "REASONING", "category": "Anomaly Detection"},
    {"id": "4.7", "question": "What correlations exist between facility characteristics that move together?", "expected": "REASONING", "category": "Anomaly Detection"},
    {"id": "4.8", "question": "Which facilities have unusually high breadth of procedures relative to infrastructure?", "expected": "REASONING", "category": "Anomaly Detection"},
    {"id": "4.9", "question": "Where do we see things that shouldn't move together?", "expected": "REASONING", "category": "Anomaly Detection"},
    {"id": "6.1", "question": "Where is the workforce for surgery actually practicing in Ghana?", "expected": "REASONING", "category": "Workforce"},
    {"id": "7.5", "question": "In each region, which procedures depend on very few facilities?", "expected": "STRUCTURED", "category": "Resource Gaps"},
    {"id": "7.6", "question": "Where is there oversupply of low-complexity procedures versus scarcity of high-complexity procedures?", "expected": "REASONING", "category": "Resource Gaps"},
    {"id": "8.3", "question": "Where are there gaps where no organizations are currently working despite evident need?", "expected": "DESERT", "category": "NGO Analysis"},
]

print(f"Running {len(MUST_HAVE_QUESTIONS)} Must-Have questions...\n")

Running 15 Must-Have questions...



## 10F. Execute All Tests

In [0]:
results = []
for i, mhq in enumerate(MUST_HAVE_QUESTIONS):
    print(f"\n{'='*70}")
    print(f"[{i+1}/{len(MUST_HAVE_QUESTIONS)}] Q{mhq['id']}: {mhq['question'][:65]}...")
    print(f"{'='*70}")

    with mlflow.start_run(run_name=f"eval_{mhq['id']}"):
        result = traced_query(mhq["question"])
        correct_route = result["category"] == mhq["expected"]

        mlflow.log_param("eval_id", mhq["id"])
        mlflow.log_param("category_expected", mhq["expected"])
        mlflow.log_param("category_actual", result["category"])
        mlflow.log_param("correct_route", correct_route)
        mlflow.log_metric("response_time", result["time"])
        mlflow.log_text(result["answer"], "answer.txt")

        results.append({
            "id": mhq["id"],
            "category": mhq["category"],
            "question": mhq["question"],
            "expected_route": mhq["expected"],
            "actual_route": result["category"],
            "correct_route": correct_route,
            "agent_used": result["agent"],
            "answer_length": len(result["answer"]),
            "time_seconds": result["time"],
            "answer_preview": result["answer"][:300]
        })

        status = "✓" if correct_route else "✗"
        print(f"  Route: {status} (expected={mhq['expected']}, actual={result['category']})")
        print(f"  Time: {result['time']}s | Answer: {len(result['answer'])} chars")
        print(f"  Preview: {result['answer'][:150]}...")


[1/15] Q1.1: How many hospitals have cardiology?...
  Route: ✓ (expected=STRUCTURED, actual=STRUCTURED)
  Time: 2.81s | Answer: 646 chars
  Preview: That's a pretty broad question.  There are thousands of hospitals around the world, and many of them have cardiology departments. It's difficult to gi...

[2/15] Q1.2: How many hospitals in Greater Accra have the ability to perform s...
  Route: ✓ (expected=STRUCTURED, actual=STRUCTURED)
  Time: 2.56s | Answer: 473 chars
  Preview: According to the data, there are approximately 12 hospitals in the Greater Accra region that have the capability to perform surgery. However, please n...

[3/15] Q1.3: What services does Korle Bu Teaching Hospital offer?...
  Route: ✗ (expected=SEMANTIC, actual=STRUCTURED)
  Time: 2.3s | Answer: 609 chars
  Preview: Korle Bu Teaching Hospital is a major health facility in Ghana, and it offers a wide range of medical services. These include general outpatient and i...

[4/15] Q1.4: Are there any clinics in Tamal

## 10G. Results Summary

In [0]:
correct = sum(1 for r in results if r["correct_route"])
total = len(results)
avg_time = sum(r["time_seconds"] for r in results) / total
avg_length = sum(r["answer_length"] for r in results) / total

print("=" * 70)
print("EVALUATION RESULTS SUMMARY")
print("=" * 70)
print(f"""
Total Questions:    {total}
Correct Routing:    {correct}/{total} ({100*correct/total:.1f}%)
Average Time:       {avg_time:.2f}s
Average Answer Len: {avg_length:.0f} chars

BY CATEGORY:
""")

from collections import defaultdict
by_cat = defaultdict(list)
for r in results:
    by_cat[r["category"]].append(r)

for cat, items in sorted(by_cat.items()):
    cat_correct = sum(1 for i in items if i["correct_route"])
    cat_avg_time = sum(i["time_seconds"] for i in items) / len(items)
    print(f"  {cat:20s}: {cat_correct}/{len(items)} correct, avg {cat_avg_time:.2f}s")

print(f"\nDETAILED RESULTS:")
print(f"{'ID':>5} | {'Expected':>12} | {'Actual':>12} | {'OK':>3} | {'Time':>6} | Question")
print("-" * 90)
for r in results:
    ok = "✓" if r["correct_route"] else "✗"
    print(f"{r['id']:>5} | {r['expected_route']:>12} | {r['actual_route']:>12} | {ok:>3} | {r['time_seconds']:>5.1f}s | {r['question'][:45]}")

EVALUATION RESULTS SUMMARY

Total Questions:    15
Correct Routing:    6/15 (40.0%)
Average Time:       2.87s
Average Answer Len: 1361 chars

BY CATEGORY:

  Anomaly Detection   : 0/4 correct, avg 2.88s
  Basic Queries       : 4/5 correct, avg 2.76s
  Geospatial          : 0/2 correct, avg 2.47s
  NGO Analysis        : 0/1 correct, avg 2.92s
  Resource Gaps       : 2/2 correct, avg 3.00s
  Workforce           : 0/1 correct, avg 3.94s

DETAILED RESULTS:
   ID |     Expected |       Actual |  OK |   Time | Question
------------------------------------------------------------------------------------------
  1.1 |   STRUCTURED |   STRUCTURED |   ✓ |   2.8s | How many hospitals have cardiology?
  1.2 |   STRUCTURED |   STRUCTURED |   ✓ |   2.6s | How many hospitals in Greater Accra have the 
  1.3 |     SEMANTIC |   STRUCTURED |   ✗ |   2.3s | What services does Korle Bu Teaching Hospital
  1.4 |     SEMANTIC |     SEMANTIC |   ✓ |   1.9s | Are there any clinics in Tamale that do surge
  1.

## 10H. Save Results to Delta Table

In [0]:
schema = StructType([
    StructField("id", StringType()), StructField("category", StringType()),
    StructField("question", StringType()), StructField("expected_route", StringType()),
    StructField("actual_route", StringType()), StructField("correct_route", StringType()),
    StructField("agent_used", StringType()), StructField("answer_length", IntegerType()),
    StructField("time_seconds", FloatType()), StructField("answer_preview", StringType()),
])

records = [{**r, "correct_route": str(r["correct_route"])} for r in results]
spark.createDataFrame(records, schema=schema).write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(RESULTS_TABLE)
print(f"Results saved to {RESULTS_TABLE}")
display(spark.table(RESULTS_TABLE))

Results saved to hackathon_vf.healthcare.evaluation_results


id,category,question,expected_route,actual_route,correct_route,agent_used,answer_length,time_seconds,answer_preview
1.1,Basic Queries,How many hospitals have cardiology?,STRUCTURED,STRUCTURED,True,sql,646,2.81,"That's a pretty broad question. There are thousands of hospitals around the world, and many of them have cardiology departments. It's difficult to give an exact number, but I can tell you that most major hospitals and many smaller ones have some level of cardiology care. In the United States alone"
1.2,Basic Queries,How many hospitals in Greater Accra have the ability to perform surgery?,STRUCTURED,STRUCTURED,True,sql,473,2.56,"According to the data, there are approximately 12 hospitals in the Greater Accra region that have the capability to perform surgery. However, please note that this information may be subject to change and it's always best to verify with the hospitals directly for the most up-to-date information. Som"
1.3,Basic Queries,What services does Korle Bu Teaching Hospital offer?,SEMANTIC,STRUCTURED,False,sql,609,2.3,"Korle Bu Teaching Hospital is a major health facility in Ghana, and it offers a wide range of medical services. These include general outpatient and inpatient services, emergency care, surgical services, maternity and child health services, laboratory and diagnostic services, and specialized service"
1.4,Basic Queries,Are there any clinics in Tamale that do surgery?,SEMANTIC,SEMANTIC,True,rag,956,1.86,"According to the data, there are several clinics and hospitals in Tamale that offer surgical services. For example, Ospedale Didattico di Tamale (hospital) [1] offers various surgical specialties such as ""generalSurgery"", ""neurosurgery"", ""oralAndMaxillofacialSurgery"", ""orthopedicSurgery"", and ""uro"
1.5,Basic Queries,Which region has the most hospital-type facilities?,STRUCTURED,STRUCTURED,True,sql,423,4.28,"Based on general knowledge, the region with the most hospital-type facilities is likely to be North America, particularly the United States. The US has a high number of hospitals and medical facilities, with many of them being well-equipped and technologically advanced. However, without specific dat"
2.1,Geospatial,Which regions have hospitals that can treat cardiac conditions?,STRUCTURED,SEMANTIC,False,rag,894,1.71,"Based on the provided data, the regions with hospitals that can treat cardiac conditions (cardiology) are: 1. Northern Region - No hospital with cardiology specs found. 2. Volta Region - No hospital with cardiology specs found. 3. Upper East Region - Eastside Life Hospital has cardiology specs. 4."
2.3,Geospatial,Where are the largest geographic cold spots where critical procedures are absent?,DESERT,STRUCTURED,False,sql,1881,3.23,"The largest geographic cold spots where critical procedures are absent can be found in various parts of the world, particularly in low- and middle-income countries. Some of the most notable regions include: 1. Sub-Saharan Africa: Countries such as the Democratic Republic of Congo, Ethiopia, and Nig"
4.4,Anomaly Detection,Which facilities claim an unrealistic number of procedures relative to their size?,REASONING,STRUCTURED,False,sql,268,3.5,"Based on the data, it appears that facility A claims an unrealistic number of procedures relative to their size. Can I see the specific numbers to confirm this? What are the actual procedure numbers and what is considered a realistic range for a facility of that size?"
4.7,Anomaly Detection,What correlations exist between facility characteristics that move together?,REASONING,STRUCTURED,False,sql,2165,3.78,"To identify correlations between facility characteristics that move together, we would typically look for patterns or relationships within the data provided. However, since the specific data isn't mentioned in your query, I'll outline a general approach to how one might analyze such correlations. 1"
4.8,Anomaly Detection,Which facilities have unusually high breadth of procedures relative to infrastruct

## 10I. Project Summary Dashboard

In [0]:
summary_html = f"""
<html>
<head><style>
body {{ background: #1a1a2e; color: #ecf0f1; font-family: 'Segoe UI', sans-serif; padding: 30px; }}
.title {{ text-align: center; font-size: 32px; font-weight: bold; margin-bottom: 5px; }}
.subtitle {{ text-align: center; font-size: 16px; color: #bdc3c7; margin-bottom: 30px; }}
.stats {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin-bottom: 30px; }}
.card {{ background: linear-gradient(135deg, #2c3e50, #34495e); border-radius: 12px; padding: 20px; text-align: center; }}
.card .value {{ font-size: 36px; font-weight: bold; }}
.card .label {{ font-size: 12px; color: #95a5a6; text-transform: uppercase; }}
.section {{ background: #16213e; border-radius: 12px; padding: 20px; margin-bottom: 20px; }}
.section h3 {{ color: #3498db; margin-top: 0; }}
table {{ width: 100%; border-collapse: collapse; }}
th, td {{ padding: 8px 12px; text-align: left; border-bottom: 1px solid #2c3e50; }}
th {{ color: #3498db; }}
.pass {{ color: #2ecc71; }} .fail {{ color: #e74c3c; }}
</style></head>
<body>
<div class="title">🏥 Healthcare Intelligence Agent</div>
<div class="subtitle">Virtue Foundation — Final Evaluation Report</div>

<div class="stats">
    <div class="card"><div class="label">Facilities</div><div class="value" style="color:#3498db;">{len(facilities_pdf)}</div></div>
    <div class="card"><div class="label">Questions Tested</div><div class="value" style="color:#f39c12;">{total}</div></div>
    <div class="card"><div class="label">Routing Accuracy</div><div class="value" style="color:{'#2ecc71' if correct/total > 0.8 else '#e74c3c'};">{100*correct/total:.0f}%</div></div>
    <div class="card"><div class="label">Avg Response</div><div class="value" style="color:#9b59b6;">{avg_time:.1f}s</div></div>
</div>

<div class="section">
    <h3>📋 Must-Have Question Results</h3>
    <table>
        <tr><th>ID</th><th>Category</th><th>Expected</th><th>Actual</th><th>Status</th><th>Time</th></tr>
        {"".join(f'<tr><td>{r["id"]}</td><td>{r["category"]}</td><td>{r["expected_route"]}</td><td>{r["actual_route"]}</td><td class="{"pass" if r["correct_route"] else "fail"}">{"✓" if r["correct_route"] else "✗"}</td><td>{r["time_seconds"]:.1f}s</td></tr>' for r in results)}
    </table>
</div>

<div class="section">
    <h3>🏗️ Architecture Components</h3>
    <table>
        <tr><th>Component</th><th>Technology</th><th>Status</th></tr>
        <tr><td>Data Storage</td><td>Delta Tables (Unity Catalog)</td><td class="pass">✓ Active</td></tr>
        <tr><td>Text-to-SQL</td><td>Databricks LLM + Spark SQL</td><td class="pass">✓ Active</td></tr>
        <tr><td>Vector Search</td><td>Databricks VS / FAISS fallback</td><td class="pass">✓ Active</td></tr>
        <tr><td>Reasoning Agent</td><td>Databricks Llama 3.3 70B</td><td class="pass">✓ Active</td></tr>
        <tr><td>Supervisor Router</td><td>LangGraph State Machine</td><td class="pass">✓ Active</td></tr>
        <tr><td>MLflow Tracing</td><td>Step-level logging + citations</td><td class="pass">✓ Active</td></tr>
        <tr><td>Visualization</td><td>Folium Maps + Matplotlib</td><td class="pass">✓ Active</td></tr>
    </table>
</div>

<div class="section">
    <h3>🌍 Social Impact</h3>
    <p>This system helps the Virtue Foundation:</p>
    <ul>
        <li>Identify <b>medical deserts</b> — regions where people cannot access critical healthcare</li>
        <li>Detect <b>anomalous facility claims</b> — unrealistic procedures, equipment mismatches</li>
        <li>Prioritize <b>resource allocation</b> — where to send doctors, equipment, and funding</li>
        <li>Provide <b>transparent citations</b> — every answer traces back to specific facility data</li>
    </ul>
</div>
</body></html>
"""
displayHTML(summary_html)

ID,Category,Expected,Actual,Status,Time
1.1,Basic Queries,STRUCTURED,STRUCTURED,✓,2.8s
1.2,Basic Queries,STRUCTURED,STRUCTURED,✓,2.6s
1.3,Basic Queries,SEMANTIC,STRUCTURED,✗,2.3s
1.4,Basic Queries,SEMANTIC,SEMANTIC,✓,1.9s
1.5,Basic Queries,STRUCTURED,STRUCTURED,✓,4.3s
2.1,Geospatial,STRUCTURED,SEMANTIC,✗,1.7s
2.3,Geospatial,DESERT,STRUCTURED,✗,3.2s
4.4,Anomaly Detection,REASONING,STRUCTURED,✗,3.5s
4.7,Anomaly Detection,REASONING,STRUCTURED,✗,3.8s
4.8,Anomaly Detection,REASONING,SEMANTIC,✗,2.5s


## 10J. Quick Demo Function

In [0]:
def demo(question: str):
    """Demo function for live presentation — clean output with timing."""
    print(f"\n🔍 Question: {question}")
    print("─" * 60)
    result = traced_query(question)
    print(f"🏷️ Category: {result['category']} → Agent: {result['agent']}")
    print(f"⏱️ Response time: {result['time']}s")
    print(f"\n📝 Answer:\n{result['answer']}")
    print("─" * 60)
    return result

## Demo Questions

```python
demo("How many hospitals have cardiology?")
demo("What services does Korle Bu Teaching Hospital offer?")
demo("Which facilities have suspicious procedure claims?")
demo("Where are the biggest medical deserts in Ghana?")
demo("Where should the Virtue Foundation send volunteer surgeons?")
```

In [0]:
print("=" * 60)
print("PROJECT COMPLETE!")
print("=" * 60)
print(f"""
Healthcare Intelligence Agent for the Virtue Foundation

Pipeline:
  00_setup           → Environment & API keys
  01_data_cleaning   → Deduplication ({len(facilities_pdf)} clean facilities)
  02_data_analysis   → Anomaly flags & regional stats
  03_vector_store    → Databricks Vector Search / FAISS
  04_rag_chain       → RAG pipeline with citations
  05_sql_agent       → Text-to-SQL structured queries
  06_reasoning_agent → Medical reasoning & anomaly detection
  07_supervisor      → LangGraph multi-agent router
  08_dashboard       → Maps & visualization
  09_mlflow_tracing  → Step-level tracing & citations
  10_final_testing   → Evaluation ({correct}/{total} routing accuracy)

""")

PROJECT COMPLETE!

Healthcare Intelligence Agent for the Virtue Foundation

Pipeline:
  00_setup           → Environment & API keys
  01_data_cleaning   → Deduplication (797 clean facilities)
  02_data_analysis   → Anomaly flags & regional stats
  03_vector_store    → Databricks Vector Search / FAISS
  04_rag_chain       → RAG pipeline with citations
  05_sql_agent       → Text-to-SQL structured queries
  06_reasoning_agent → Medical reasoning & anomaly detection
  07_supervisor      → LangGraph multi-agent router
  08_dashboard       → Maps & visualization
  09_mlflow_tracing  → Step-level tracing & citations
  10_final_testing   → Evaluation (6/15 routing accuracy)

Ready for submission! 🚀

